# Global D2+ Expected Matchup Collection

This notebook directly collects patch-16.19 matchup pages from LoLalytics. It does **not** use `lolalytics-api`, whose hard-coded XPath no longer matches the current site. Every result is checkpointed immediately, so an interrupted run resumes safely.

In [1]:
%pip install -q requests pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib
import sys
from pathlib import Path

start_dir = Path.cwd().resolve()
search_dirs = [start_dir, *start_dir.parents]
PROJECT_DIR = next(
    (p for p in search_dirs if (p / 'data' / 'processed' / 'unique_matchups_16_19.csv').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        'Could not find data/processed/unique_matchups_16_19.csv. '
        'Put both downloaded files in the repository, then reopen this notebook there.'
    )

sys.path.insert(0, str(PROJECT_DIR))
import lolalytics_matchup_scraper as matchup_scraper
matchup_scraper = importlib.reload(matchup_scraper)

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
UNIQUE_MATCHUPS_PATH = PROCESSED_DIR / 'unique_matchups_16_19.csv'
print('Project directory:', PROJECT_DIR)
print('Unique-matchup input found:', UNIQUE_MATCHUPS_PATH.exists())

Project directory: /Users/minhho-hoang/Documents/riot_proj/riot-game-prediction
Unique-matchup input found: True


In [3]:
TARGET_PATCH = '16.19'
MATCHUP_RANK = 'd2_plus'
MATCHUP_REGIONS = ('global',)  # Collect NA later with ('global', 'na') if wanted
REQUEST_DELAY_SECONDS = 1.25
RUN_FULL_COLLECTION = True

## One-call preflight

This makes one real Global D2+ request and verifies that the current page text can be parsed before beginning the full run.

In [4]:
preflight_url, preflight = matchup_scraper.scrape_one_matchup(
    champion='Ahri',
    opponent_champion='Zed',
    lane='mid',
    patch=TARGET_PATCH,
    rank=MATCHUP_RANK,
    region='global',
)
assert preflight['status'] in {'ok', 'no_data'}
print('Preflight passed:', preflight)
print('URL:', preflight_url)

Preflight passed: {'raw_matchup_winrate': 51.19, 'tier_average_winrate': None, 'lolalytics_delta2': 0.17, 'expected_winrate': 51.19, 'matchup_games': 881, 'status': 'ok', 'error': ''}
URL: https://lolalytics.com/lol/ahri/vs/zed/build/?lane=middle&vslane=middle&tier=d2_plus&patch=16.19


## Resumable collection

You may interrupt this cell. Every attempted matchup is appended immediately to `expected_matchups_16_19_checkpoint.jsonl`. Running it again skips successful/no-data rows and retries errors.

In [7]:
if RUN_FULL_COLLECTION:
    expected_matchups = matchup_scraper.collect_expected_matchups(
        unique_matchups_path=UNIQUE_MATCHUPS_PATH,
        processed_dir=PROCESSED_DIR,
        patch=TARGET_PATCH,
        rank=MATCHUP_RANK,
        regions=MATCHUP_REGIONS,
        delay_seconds=REQUEST_DELAY_SECONDS,
    )
else:
    stem = f"expected_matchups_{TARGET_PATCH.replace('.', '_')}"
    saved_rows = matchup_scraper.load_checkpoint(PROCESSED_DIR / f'{stem}_checkpoint.jsonl')
    expected_matchups = matchup_scraper.export_results(
        saved_rows.values(),
        PROCESSED_DIR / f'{stem}.csv',
        PROCESSED_DIR / f'{stem}.parquet',
    )

Required matchup-region lookups: 7,043
Completed checkpoint rows retained: 6,886
Lookups remaining: 157
Lookup 1/157 | global | adc Azir vs Veigar | error
Lookup 2/157 | global | adc Cassiopeia vs TwistedFate | error
Lookup 3/157 | global | adc Galio vs Pyke | error
Lookup 4/157 | global | adc Gangplank vs Katarina | error
Lookup 5/157 | global | adc Garen vs Yone | error
Lookup 6/157 | global | adc Gnar vs Galio | error
Lookup 7/157 | global | adc Kaisa vs Ivern | error
Lookup 8/157 | global | adc Kalista vs Azir | error
Lookup 9/157 | global | adc KogMaw vs Akshan | error
Lookup 10/157 | global | adc KogMaw vs Ambessa | error
Lookup 11/157 | global | adc KogMaw vs Fiora | error
Lookup 12/157 | global | adc KogMaw vs Neeko | error
Lookup 13/157 | global | adc KogMaw vs TwistedFate | error
Lookup 14/157 | global | adc Locke vs Zed | error
Lookup 15/157 | global | adc Lulu vs Yasuo | error
Lookup 16/157 | global | adc Mel vs Riven | error
Collection interrupted; completed checkpoint row

## Coverage and outputs

In [6]:
display(expected_matchups.groupby(['region', 'status'], dropna=False).size().rename('rows').to_frame())
successful = expected_matchups.loc[expected_matchups['status'].eq('ok')]
if not successful.empty:
    display(successful['matchup_games'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

print('CSV:', PROCESSED_DIR / 'expected_matchups_16_19.csv')
print('Parquet:', PROCESSED_DIR / 'expected_matchups_16_19.parquet')
print('Checkpoint:', PROCESSED_DIR / 'expected_matchups_16_19_checkpoint.jsonl')

rows
region status      
global error    157
       ok      6886

count    6886.000000
mean      229.687191
std       368.555659
min         1.000000
10%         8.000000
25%        32.000000
50%       116.000000
75%       280.000000
90%       552.000000
max      4758.000000
Name: matchup_games, dtype: float64

CSV: /Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/data/processed/expected_matchups_16_19.csv
Parquet: /Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/data/processed/expected_matchups_16_19.parquet
Checkpoint: /Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/data/processed/expected_matchups_16_19_checkpoint.jsonl
